# Identification of Territorial Structures and Local Anomalies in Paraguayan Secondary Education

This notebook reproduces the analysis pipeline described in the paper: construction of district-level territorial variables, standardization, UMAP dimensionality reduction, HDBSCAN density-based clustering, LOF local anomaly detection, internal validation, baseline comparison, sensitivity analysis, and export of results.

The computational core preserves the original code structure and variables used to generate the results reported in the paper.


## 0. Environment and dependencies

Install the required packages only when they are not already available in the execution environment. The main libraries used are `pandas`, `numpy`, `scipy`, `scikit-learn`, `umap-learn`, `hdbscan`, and `matplotlib`.


In [ ]:
# Install these packages if needed. In Google Colab, this cell can be executed directly.
!pip -q install umap-learn hdbscan


In [ ]:
import os
import zipfile

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import entropy
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

import umap
import hdbscan

# Output folders for GitHub-friendly artifacts.
FIGURE_DIR = "figures"
OUTPUT_DIR = "outputs"
os.makedirs(FIGURE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 1. Data loading and initial setup

The original MEC enrollment file is loaded. Categorical names are standardized, and auxiliary variables are created for male enrollment, female enrollment, and total enrollment.

Place the CSV file in the same folder as this notebook or update the `path` variable below.


In [ ]:
path = "matriculaciones_departamentos_distritos_20260120.csv"
df = pd.read_csv(path)

# Standardize names in case of variations in uppercase, lowercase, or extra spaces.
df["nombre_zona"] = df["nombre_zona"].astype(str).str.strip()
df["sector_o_tipo_gestion"] = df["sector_o_tipo_gestion"].astype(str).str.strip()

# Convert enrollment counts to numeric values and create the total enrollment variable.
df["h"] = pd.to_numeric(df["cantidad_matriculados_hombre"], errors="coerce").fillna(0)
df["m"] = pd.to_numeric(df["cantidad_matriculados_mujer"], errors="coerce").fillna(0)
df["total"] = df["h"] + df["m"]

df.head(), df.shape


## 2. Construction of territorial variables

The paper uses the district as the unit of analysis. The following variables are constructed at the district level:

- `matricula_total`: total enrollment.
- `pct_mujeres`: proportion of female enrollment.
- `share_urbana`: proportion of enrollment in urban areas.
- `share_privado`: proportion of enrollment in privately managed institutions.
- `entropia_contexto`: contextual entropy of area-management combinations.


In [ ]:
group_cols = ["codigo_departamento", "nombre_departamento", "codigo_distrito", "nombre_distrito"]

def entropia_contexto(sub):
    # Enrollment distribution by area and management sector.
    combos = sub.groupby(["nombre_zona", "sector_o_tipo_gestion"])["total"].sum()
    if combos.sum() == 0:
        return 0.0
    p = (combos / combos.sum()).values
    return float(entropy(p, base=2))

rows = []
for keys, sub in df.groupby(group_cols):
    total = sub["total"].sum()
    h = sub["h"].sum()
    m = sub["m"].sum()

    # Smoothing is used for numerical stability.
    pct_mujeres = (m + 0.5) / (total + 1.0) if total > 0 else np.nan
    ratio_h_m = (h + 0.5) / (m + 0.5)

    urbana = sub.loc[sub["nombre_zona"].str.lower() == "urbana", "total"].sum()
    privado = sub.loc[sub["sector_o_tipo_gestion"].str.lower() == "privado", "total"].sum()

    rows.append((*keys,
                 total,
                 pct_mujeres,
                 ratio_h_m,
                 (urbana / total if total > 0 else 0.0),
                 (privado / total if total > 0 else 0.0),
                 entropia_contexto(sub)))

territ = pd.DataFrame(rows, columns=group_cols + [
    "matricula_total", "pct_mujeres", "ratio_h_m", "share_urbana", "share_privado", "entropia_contexto"
])

territ.describe(include="all")


## 3. Descriptive statistics for the analytical variables

This section summarizes the district-level variables used in the modeling pipeline. These statistics correspond to the descriptive stage reported in the paper.


In [ ]:
features = ["matricula_total", "pct_mujeres", "share_urbana", "share_privado", "entropia_contexto"]

X = territ[features].replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))

X.describe()


## 4. Standardization and nonlinear dimensionality reduction using UMAP

The five analytical variables are standardized using `StandardScaler`. Then, UMAP is applied with the same parameters used in the paper: `n_neighbors = 15`, `min_dist = 0.05`, `n_components = 2`, and `random_state = 42`.


In [ ]:
Xs = StandardScaler().fit_transform(X)

umap_model = umap.UMAP(
    n_neighbors=15,
    min_dist=0.05,
    n_components=2,
    random_state=42
)
emb = umap_model.fit_transform(Xs)

territ["u1"] = emb[:, 0]
territ["u2"] = emb[:, 1]

territ.head()


## 5. Density-based clustering with HDBSCAN

HDBSCAN is applied to the two-dimensional UMAP representation. This preserves the original computational logic used to obtain the six clusters and the noise group reported in the paper.


In [ ]:
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=12,
    min_samples=8,
    metric="euclidean"
)

labels = clusterer.fit_predict(emb)
territ["cluster"] = labels
territ["prob_cluster"] = clusterer.probabilities_

territ["cluster"].value_counts().sort_index()


## 6. Figure 2: UMAP + HDBSCAN clusters

This figure follows the original plotting code so that the visualization remains consistent with the clustering results reported in the paper. The label `-1` corresponds to noise.


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(territ["u1"], territ["u2"], c=territ["cluster"], s=25, alpha=0.9)
plt.title("UMAP + HDBSCAN (clusters by density; -1 = noise)")
plt.xlabel("u1")
plt.ylabel("u2")
plt.savefig(os.path.join(FIGURE_DIR, "figure_02_umap_hdbscan_clusters.png"), dpi=300, bbox_inches="tight")
plt.show()


## 7. Table II: Distribution of districts by cluster

The following table reports the number of districts assigned to each HDBSCAN cluster, including the noise group.


In [ ]:
cluster_distribution = (
    territor_cluster_counts := territ["cluster"].value_counts().sort_index()
).rename_axis("cluster").reset_index(name="n_districts")

cluster_distribution


## 8. Table III: Average profile of territorial variables by cluster

This table summarizes the average territorial profile of each cluster using the five analytical variables.


In [ ]:
cluster_profile = (
    territ.groupby("cluster")[features]
    .mean()
    .round(4)
    .reset_index()
)

cluster_profile


## 9. Local anomaly detection using LOF

The Local Outlier Factor is applied to the standardized feature space, as described in the paper. A contamination level of `0.05` and `n_neighbors = 20` are used.


In [ ]:
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
pred = lof.fit_predict(Xs)
score = -lof.negative_outlier_factor_

territ["lof_outlier"] = (pred == -1).astype(int)
territ["lof_score"] = score

top_out = territ.sort_values("lof_score", ascending=False).head(15)
top_out[[
    "nombre_departamento", "nombre_distrito", "cluster", "lof_score",
    "matricula_total", "share_urbana", "share_privado", "pct_mujeres"
]]


## 10. Figure 3: Local anomalies on the UMAP projection

The anomaly visualization uses the same UMAP coordinates and highlights the districts identified as local anomalies by LOF.


In [ ]:
labels_lof = lof.fit_predict(Xs)
mask_anom = labels_lof == -1

plt.figure(figsize=(8, 6))
plt.scatter(emb[~mask_anom, 0], emb[~mask_anom, 1],
            c="blue", s=25, alpha=0.6, label="Normal")
plt.scatter(emb[mask_anom, 0], emb[mask_anom, 1],
            c="red", s=40, label="Anomalies")

plt.title("UMAP (territories): 2D projection")
plt.xlabel("u1")
plt.ylabel("u2")
plt.legend()
plt.savefig(os.path.join(FIGURE_DIR, "figure_03_umap_anomalies.png"), dpi=300, bbox_inches="tight")
plt.show()


## 11. Main result checks

These checks help verify that the notebook output remains consistent with the paper: six HDBSCAN clusters, a noise group, and the LOF anomaly count.


In [ ]:
print("Clusters:")
print(territ["cluster"].value_counts().sort_index())

print("
Noise points:")
print((territ["cluster"] == -1).sum())

print("
LOF anomalies:")
print(territ["lof_outlier"].sum())


## 12. Table IV: Internal validation metrics for UMAP + HDBSCAN

The internal validation metrics are computed after excluding noise points, which is consistent with the density-based clustering approach.


In [ ]:
mask = labels != -1
emb_valid = emb[mask]
labels_valid = labels[mask]

n_clusters = len(np.unique(labels_valid))

print(f"Valid clusters excluding noise: {n_clusters}")
print(f"Non-noise points: {mask.sum()} of {len(labels)}")

if n_clusters > 1 and len(labels_valid) > n_clusters:
    sil = silhouette_score(emb_valid, labels_valid)
    db = davies_bouldin_score(emb_valid, labels_valid)
    ch = calinski_harabasz_score(emb_valid, labels_valid)

    print(f"Silhouette: {sil:.4f}")
    print(f"Davies-Bouldin: {db:.4f}")
    print(f"Calinski-Harabasz: {ch:.4f}")
else:
    sil = np.nan
    db = np.nan
    ch = np.nan
    print("There are not enough valid clusters to compute the metrics.")


In [ ]:
hdbscan_metrics = pd.DataFrame([{
    "method": "UMAP + HDBSCAN",
    "n_clusters_excluding_noise": n_clusters,
    "n_noise": int((labels == -1).sum()),
    "pct_noise": float((labels == -1).mean() * 100),
    "silhouette": sil,
    "davies_bouldin": db,
    "calinski_harabasz": ch
}])

hdbscan_metrics


## 13. Baseline comparison: PCA + K-means

As in the paper, the baseline uses PCA for two-dimensional reduction and K-means with the same number of clusters identified by HDBSCAN, excluding noise.


In [ ]:
pca = PCA(n_components=2, random_state=42)
emb_pca = pca.fit_transform(Xs)

# Use the same number of clusters detected by HDBSCAN, excluding noise.
k = len(np.unique(labels[labels != -1]))

kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
labels_kmeans = kmeans.fit_predict(emb_pca)

sil_k = silhouette_score(emb_pca, labels_kmeans)
db_k = davies_bouldin_score(emb_pca, labels_kmeans)
ch_k = calinski_harabasz_score(emb_pca, labels_kmeans)

print(f"KMeans clusters: {k}")
print(f"Silhouette: {sil_k:.4f}")
print(f"Davies-Bouldin: {db_k:.4f}")
print(f"Calinski-Harabasz: {ch_k:.4f}")


## 14. Table V: Comparison between UMAP + HDBSCAN and PCA + K-means

This table compares the proposed approach with the baseline using the same internal validation metrics.


In [ ]:
comparison = pd.DataFrame([
    {
        "method": "UMAP + HDBSCAN",
        "n_clusters": len(np.unique(labels[labels != -1])),
        "n_noise": int((labels == -1).sum()),
        "pct_noise": float((labels == -1).mean() * 100),
        "silhouette": sil,
        "davies_bouldin": db,
        "calinski_harabasz": ch
    },
    {
        "method": "PCA + KMeans",
        "n_clusters": k,
        "n_noise": 0,
        "pct_noise": 0.0,
        "silhouette": sil_k,
        "davies_bouldin": db_k,
        "calinski_harabasz": ch_k
    }
])

comparison


## 15. Figure 4: PCA + K-means baseline

This figure shows the clustering structure obtained using the baseline method.


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(emb_pca[:, 0], emb_pca[:, 1], c=labels_kmeans, s=25, alpha=0.9)
plt.title("PCA + KMeans (baseline)")
plt.xlabel("pc1")
plt.ylabel("pc2")
plt.savefig(os.path.join(FIGURE_DIR, "figure_04_pca_kmeans_baseline.png"), dpi=300, bbox_inches="tight")
plt.show()


## 16. Table VI: UMAP sensitivity analysis

This section evaluates selected UMAP configurations while keeping the same HDBSCAN parameters. The selected configuration used in the paper is `n_neighbors = 15` and `min_dist = 0.05`.


In [ ]:
sens_results = []

for n_neighbors in [10, 15, 20, 25]:
    for min_dist in [0.01, 0.05, 0.1]:
        emb_tmp = umap.UMAP(
            n_neighbors=n_neighbors,
            min_dist=min_dist,
            n_components=2,
            random_state=42
        ).fit_transform(Xs)

        labels_tmp = hdbscan.HDBSCAN(
            min_cluster_size=12,
            min_samples=8,
            metric="euclidean"
        ).fit_predict(emb_tmp)

        mask_tmp = labels_tmp != -1
        unique_tmp = np.unique(labels_tmp[mask_tmp])
        n_clusters_tmp = len(unique_tmp)
        n_noise_tmp = int((labels_tmp == -1).sum())

        if n_clusters_tmp > 1 and mask_tmp.sum() > n_clusters_tmp:
            sil_tmp = silhouette_score(emb_tmp[mask_tmp], labels_tmp[mask_tmp])
            db_tmp = davies_bouldin_score(emb_tmp[mask_tmp], labels_tmp[mask_tmp])
            ch_tmp = calinski_harabasz_score(emb_tmp[mask_tmp], labels_tmp[mask_tmp])
        else:
            sil_tmp = np.nan
            db_tmp = np.nan
            ch_tmp = np.nan

        sens_results.append({
            "n_neighbors": n_neighbors,
            "min_dist": min_dist,
            "n_clusters": n_clusters_tmp,
            "n_noise": n_noise_tmp,
            "pct_noise": 100 * n_noise_tmp / len(labels_tmp),
            "silhouette": sil_tmp,
            "davies_bouldin": db_tmp,
            "calinski_harabasz": ch_tmp
        })

sens_df = pd.DataFrame(sens_results)
sens_df.sort_values(["silhouette", "davies_bouldin"], ascending=[False, True])


## 17. Exporting results for GitHub

The processed datasets, cluster profiles, validation metrics, sensitivity results, and figures are saved in organized folders.


In [ ]:
# Save tabular outputs.
territ.to_csv(os.path.join(OUTPUT_DIR, "territorial_results.csv"), index=False)
cluster_distribution.to_csv(os.path.join(OUTPUT_DIR, "cluster_distribution.csv"), index=False)
cluster_profile.to_csv(os.path.join(OUTPUT_DIR, "cluster_profile.csv"), index=False)
top_out.to_csv(os.path.join(OUTPUT_DIR, "top_lof_outliers.csv"), index=False)
hdbscan_metrics.to_csv(os.path.join(OUTPUT_DIR, "hdbscan_internal_validation_metrics.csv"), index=False)
comparison.to_csv(os.path.join(OUTPUT_DIR, "baseline_comparison_metrics.csv"), index=False)
sens_df.to_csv(os.path.join(OUTPUT_DIR, "umap_sensitivity_analysis.csv"), index=False)

# Save an Excel workbook for manual review.
with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "territorial_analysis_results.xlsx")) as writer:
    territ.to_excel(writer, sheet_name="Complete_results", index=False)
    cluster_distribution.to_excel(writer, sheet_name="Cluster_distribution", index=False)
    cluster_profile.to_excel(writer, sheet_name="Cluster_profile", index=False)
    top_out.to_excel(writer, sheet_name="Top_LOF_outliers", index=False)
    hdbscan_metrics.to_excel(writer, sheet_name="HDBSCAN_metrics", index=False)
    comparison.to_excel(writer, sheet_name="Baseline_comparison", index=False)
    sens_df.to_excel(writer, sheet_name="Sensitivity_analysis", index=False)

print("Results exported successfully.")


In [ ]:
# Optional: create a ZIP file with the generated outputs and figures.
zip_filename = "territorial_analysis_umap_hdbscan_lof_outputs.zip"

with zipfile.ZipFile(zip_filename, "w") as zipf:
    for folder in [OUTPUT_DIR, FIGURE_DIR]:
        for root, _, files in os.walk(folder):
            for file in files:
                file_path = os.path.join(root, file)
                zipf.write(file_path, arcname=file_path)

print("ZIP file generated:", zip_filename)
